## Preliminares

In [1]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.model_selection import cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import plotly.express as px
from src.evaluators import *

In [2]:
# Abrir archivo clean_data
data_folder = "data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   MarketCap                   17986 non-null  float64 
 1   EnterpriseValue             17986 non-null  float64 
 2   PE_Trailing                 17986 non-null  float64 
 3   EnterpriseToEbitda          17986 non-null  float64 
 4   PriceToBook                 17986 non-null  float64 
 5   TotalAssets                 17986 non-null  float64 
 6   TotalRevenue                17986 non-null  float64 
 7   TotalEquity                 17986 non-null  float64 
 8   Date                        17986 non-null  object  
 9   Ticker                      17986 non-null  object  
 10  Close_log                   17986 non-null  float64 
 11  Sector                      17986 non-null  category
 12  YearsSinceAdded             17986 non-null  float64 
 13  Monthly_Return  

## Feature Engineering

Sección para crear variables en la fase de modelado. 
La mayor parte de las variables fue creada en la fase de extracción.

In [3]:
# Calcular las aceleraciones netas (Momento - Tendencia) para variables de crecimiento.
# Una vez calculadas, se eliminan las variables trimestrales (reemplazar=True)
variables_de_crecimiento = ['Revenue_QoQ', 'EBITDA_QoQ', 'FCF_QoQ', 'Capex_QoQ']
df = calcular_acceleration_features(df, variables_de_crecimiento, reemplazar= True)

In [4]:
# Sectores poco significativos: se agrupan en la categoria "Other"
# Se saltea en la versión actual
#sectores_importantes = ['InformationTechnology', 'Energy']

#df['Sector'] = np.where(df['Sector'].isin(sectores_importantes), df['Sector'], 'Other')

# Se vuelve a convertir en category
#df['Sector'] = df['Sector'].astype('category')

# Modelo de ensamblado de árboles RandomForest

In [5]:
# Se asegura el ordenamiento por fecha
df.sort_values(by='Date', inplace=True)

# Eliminar predictores
predictores_a_eliminar = ['PriceToBook', 
                          'PE_Trailing',
                          'EnterpriseToEbitda', 
                          'Close_log',
                          'MarketCap',
                          'EnterpriseValue',
                          'TotalEquity',
                          'TotalAssets',
                          'TotalRevenue',
                          'Ticker',
                          'Date',
                          'Monthly_Return', # Baja relevancia
                          'Market_Covariance' # Baja relevancia
                          ]

# Se define la variable objetivo y las variables predictoras
label = 'Close_log'
X = df.drop(columns=predictores_a_eliminar) 
y = df[label]

# Columnas numéricas: 
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

# Variables categóricas:
categorical_cols = ['Sector']

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features='sqrt',
        max_samples= 0.8,
        min_samples_split= 10         
        ))
])

print("Entrenando el modelo con datos completos...")
pipe.fit(X, y)
r2_completo = pipe.score(X, y)
print(f"Entrenamiento finalizado. R2 en datos completos: {r2_completo:.4f}")

Entrenando el modelo con datos completos...
Entrenamiento finalizado. R2 en datos completos: 0.7677


In [6]:
# Test de validación cruzada
# Configurar el validador de series temporales
tscv = TimeSeriesSplit(n_splits=5) # n_splits=5 creará 5 particiones temporales secuenciales

# 3. Test de validación cruzada temporal
cross_val_scores = cross_val_score(
    estimator=pipe, 
    X=X, 
    y=y, 
    cv=tscv,         
    scoring='r2',
    n_jobs=-1        
)

print(f"R² promedio Time Series CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio Time Series CV: 0.5527 ± 0.0578


In [7]:
# Importancia de factores en el modelo
rf_model = pipe.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
4,num__ReturnOnAssets,0.118847
9,num__NetDebt_to_EBITDA,0.075635
12,num__returnOnEquity_Transformed,0.062572
3,num__profitMargins,0.055506
14,num__RelativeAssets_log,0.054885
13,num__debtToEquity_Transformed,0.054825
11,num__Capex_to_Revenue,0.052858
2,num__operatingMargins,0.049963
17,num__currentRatio_log,0.044082
10,num__FCF_to_EBITDA,0.042657


In [8]:
feature_importance_df.tail(20)

,feature,importance
15,num__RelativeRevenue_log,0.035997
18,num__Revenue_Acceleration,0.034128
28,cat__Sector_Industrials,0.020360
25,cat__Sector_Energy,0.017708
19,num__EBITDA_Acceleration,0.016968
6,num__EBITDA_YoY,0.016829
20,num__FCF_Acceleration,0.016188
24,cat__Sector_ConsumerStaples,0.015369
7,num__FCF_YoY,0.015014
21,num__Capex_Acceleration,0.014693


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Positive_Bias: si los residuos son mayores o iguales a cero.
* Negative_Bias: si los residuos son menores a cero.

In [9]:
# Se dividen los datos para predecir el valor de la última fecha disponible para cada ticker en el conjunto de test
X_train = X.iloc[:-len(df['Ticker'].unique())]  # Todos menos la última fecha de cada ticker
y_train = y.iloc[:-len(df['Ticker'].unique())]
X_test = X.iloc[-len(df['Ticker'].unique()):]   # Solo la última fecha de cada ticker
y_test = y.iloc[-len(df['Ticker'].unique()):]

# Recuperar el ticker usando el indice de y_test
tickers_test = df.loc[y_test.index, 'Ticker']

# Predicciones en el conjunto de test
y_pred = pipe.predict(X_test)

# Consolidar resultados individuales en un DataFrame
resultados_df = pd.DataFrame({
    'Ticker': tickers_test,
    'Predicho': y_pred,
    'Real': y_test
})

# Calcular el residuo para cada predicción
resultados_df['Residuo'] = resultados_df['Predicho'] - resultados_df['Real']

# Agrupar por ticker
resultados_agrupados = resultados_df.groupby('Ticker')[['Predicho', 'Real', 'Residuo']].mean()

# Generar el Cluster sobre el promedio de los residuos
resultados_agrupados['Cluster'] = ['Positive_Bias' if r >= 0 else 'Negative_Bias' 
                                   for r in resultados_agrupados['Residuo']]

# Visualizar
fig = px.scatter(
    resultados_agrupados, 
    x='Real', 
    y='Predicho', 
    color='Cluster',
    hover_name=resultados_agrupados.index, 
    labels={'Real':'Valor Real Promedio', 'Predicho':'Predicción Promedio', 'Cluster':'Sesgo del Modelo'},
    title='Predicciones vs Reales (Agrupado por Ticker)'
)

# Línea de identidad perfecta
min_val = resultados_agrupados['Real'].min()
max_val = resultados_agrupados['Real'].max()
fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
              line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster a nivel Ticker
over_mask = resultados_agrupados['Cluster'] == 'Positive_Bias'
under_mask = resultados_agrupados['Cluster'] == 'Negative_Bias'

print("\nEstadísticas por cluster (a nivel de Ticker):")
print(f"Overprediction: {over_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[over_mask, 'Residuo'].mean():.4f}")
print(f"Underprediction: {under_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[under_mask, 'Residuo'].mean():.4f}")


Estadísticas por cluster (a nivel de Ticker):
Overprediction: 189 tickers, residuo medio global: 0.4179
Underprediction: 264 tickers, residuo medio global: -0.5085


In [10]:
# Ordenar resultados por residuos
resultados_agrupados = resultados_agrupados.sort_values(by='Residuo', ascending=False)
# Positive Biases: Sobrepredicciones seleccionando solo valores positivos para los ratios Reales y Predichos
positive_biases = resultados_agrupados[(resultados_agrupados['Real'] > 0) & (resultados_agrupados['Predicho'] > 0) & (resultados_agrupados['Cluster'] == 'Positive_Bias')]

positive_biases.head(20)

,Predicho,Real,Residuo,Cluster
Ticker,,,,
GEN,4.942837,3.233961,1.708875,Positive_Bias
TTD,4.584831,3.002460,1.582371,Positive_Bias
CMG,4.766232,3.514898,1.251334,Positive_Bias
BSX,5.071659,3.866601,1.205058,Positive_Bias
PYPL,4.980610,3.786482,1.194128,Positive_Bias
CMCSA,4.338351,3.229024,1.109327,Positive_Bias
TSCO,4.569664,3.472122,1.097542,Positive_Bias
KVUE,4.058147,2.961400,1.096747,Positive_Bias
CAG,3.778517,2.699346,1.079171,Positive_Bias


In [11]:
# Establer Ticker como columna para exportar resultados
resultados_agrupados.reset_index(inplace=True)
resultados_agrupados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 453 entries, 0 to 452
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Ticker    453 non-null    object 
 1   Predicho  453 non-null    float64
 2   Real      453 non-null    float64
 3   Residuo   453 non-null    float64
 4   Cluster   453 non-null    object 
dtypes: float64(3), object(2)
memory usage: 17.8+ KB


In [12]:
# Revertir la transformación log1p a precios lineales para el reporte
resultados_agrupados['Precio_Predicho'] = np.expm1(resultados_agrupados['Predicho'])
resultados_agrupados['Precio_Real'] = np.expm1(resultados_agrupados['Real'])

# Cálculo del error en porcentaje
resultados_agrupados['Error_Porcentual'] = (resultados_agrupados['Precio_Predicho'] - resultados_agrupados['Precio_Real'])/ resultados_agrupados['Precio_Real']

resultados_agrupados.drop(columns=['Predicho', 'Real', 'Residuo'], inplace=True)

In [13]:
# Se genera el reporte final para research posterior
# Se filtra df para mantener solo la fila más reciente de cada empresa
df_latest = df.drop_duplicates(subset=['Ticker'], keep='last')

df_reporte = resultados_agrupados.merge(df_latest, how='left', on='Ticker') 

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

os.makedirs(f"{data_folder}/results", exist_ok=True) # Crear carpeta si no existe

df_reporte.to_csv(f"{data_folder}/results/reporte_{year}_{mes}_{dia}.csv", index=False)
print(f'Reporte exportado en la carpeta {data_folder}/results.')
df_reporte.head()

Reporte exportado en la carpeta data/results.


,Ticker,Cluster,Precio_Predicho,Precio_Real,Error_Porcentual,MarketCap,EnterpriseValue,PE_Trailing,EnterpriseToEbitda,PriceToBook,...,returnOnEquity_Transformed,debtToEquity_Transformed,RelativeAssets_log,RelativeRevenue_log,RelativeEquity_log,currentRatio_log,Revenue_Acceleration,EBITDA_Acceleration,FCF_Acceleration,Capex_Acceleration
0,GEN,Positive_Bias,139.167279,24.379999,4.708256,15.042459,22.900459,15.4599,8.9003,5.7612,...,0.074182,0.164644,0.000614,0.000308,0.000288,0.335686,-0.270628,-0.269880,-0.262692,0.466424
1,TTD,Positive_Bias,96.986592,19.135000,4.068544,9.343200,9.121355,21.0763,12.9362,3.7608,...,-0.012091,-0.082862,0.000242,0.000179,0.000274,0.959657,-0.184637,-0.369938,-0.237925,0.959024
2,CMG,Positive_Bias,116.475741,32.612499,2.571506,43.613869,48.339138,28.3989,20.3603,15.4080,...,0.149520,0.049439,0.000354,0.000736,0.000312,0.804107,-0.054051,-0.023155,0.042262,0.122285
3,BSX,Positive_Bias,158.438557,46.779701,2.386908,69.252670,78.723670,23.8967,15.4300,2.8579,...,-0.038227,-0.059058,0.001719,0.001238,0.002666,0.961952,-0.198643,-0.323023,-0.439163,0.055832
4,PYPL,Positive_Bias,144.563164,43.101002,2.354056,41.333861,43.271861,7.8987,5.6226,2.0406,...,0.023402,-0.057354,0.003153,0.002045,0.002229,0.827110,-0.043223,-0.141222,0.177771,0.247195


## Anexo: optimización de hiperparámetros

In [14]:
ejecutar_celda = False

if ejecutar_celda:
    nombre_modelo = "Random Forest"
    print(f"Configurando GridSearchCV para {nombre_modelo}")

    # Pipeline usando el preprocesador específico para Random Forest
    modelo_base = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('rf', RandomForestRegressor(random_state=42))
    ])

    param_grid = {
        'rf__n_estimators': [300],
        'rf__max_depth': [10],
        'rf__min_samples_leaf': [10],
        'rf__min_samples_split': [10],
        'rf__max_samples': [0.8],
        'rf__max_features': ['sqrt']
    }

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='r2',
        cv=tscv,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con datos completos
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X, y)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)